# graph-fraud-benchmark — Kaggle Benchmarking Challenge

**Question:** can an LLM catch graph-structured fraud the way a trained GNN does, when all it gets is a text description of the transaction network?

This notebook is the stage — every other file in the repo (`sample_subgraphs.py`, `serialize.py`, `models.py`, `tasks.py`, `assertions.py`, `analysis.py`) gets imported and run here, in order, exactly like actors walking on stage when it's their turn. Nothing new is implemented in this file; it only calls what already exists and is already tested.

Full design doc: `kaggle-benchmarking-challenge-architecture.md` in the repo root.

**Before running on Kaggle:** attach the IEEE-CIS Fraud Detection dataset to this notebook and confirm the exact input path in the Setup cell below — it varies depending on which dataset version you attach.

In [ ]:
# --- Setup ---
import sys
from pathlib import Path

# Make the repo's src/ importable. On Kaggle, this notebook typically
# lives at /kaggle/working/notebooks/, with the repo copied alongside
# it or added as a Kaggle Dataset/Utility Script — adjust this path to
# wherever src/ actually ends up once this is uploaded.
REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

# VERIFY ME: confirm this matches whichever IEEE-CIS dataset version
# you attach to the notebook (Add Data -> search "ieee-fraud-detection").
RAW_DATA_DIR = Path("/kaggle/input/ieee-fraud-detection")
TRANSACTIONS_CSV = RAW_DATA_DIR / "train_transaction.csv"
IDENTITY_CSV = RAW_DATA_DIR / "train_identity.csv"

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("repo root:", REPO_ROOT)
print("transactions csv exists:", TRANSACTIONS_CSV.exists())


## Step 1 — Sample subgraphs

Calls `sample_subgraphs.run()` directly rather than shelling out, so this cell's variables (the entity columns, the real per-entity-type hub caps confirmed against graph-fraud-ai's own production model) are exactly the tested code from the repo, not a reimplementation.

In [ ]:
from src.graph_fraud_benchmark.sample_subgraphs import run as sample_subgraphs_run, HUB_DEGREE_CAPS

sample_subgraphs_run(
    transactions_csv=TRANSACTIONS_CSV,
    identity_csv=IDENTITY_CSV if IDENTITY_CSV.exists() else None,
    out_dir=DATA_DIR,
    n_seeds_per_class=50,       # 50 fraud + 50 legit = 100 total samples
    hops=2,
    max_neighbors_per_entity=4,
    max_subgraph_size=12,
    hub_degree_caps=HUB_DEGREE_CAPS,   # real production thresholds, not a guess
    seed=42,
)


## Step 2 — Serialize into prompts

Label never passed in — see `serialize.py`'s own docstring for why that's structurally guaranteed, not just a promise.

In [ ]:
from src.graph_fraud_benchmark.serialize import run as serialize_run

serialize_run(
    subgraphs_dir=DATA_DIR / "subgraphs",
    out_jsonl=DATA_DIR / "serialized_subgraphs.jsonl",
)


## Step 3 — Model lineup

`kaggle_model_id` values are placeholders — confirm against Kaggle's live model catalog (the model picker in this notebook, or https://kaggle.com/benchmarks) before the real run.

In [ ]:
from src.graph_fraud_benchmark.models import get_model_lineup

MODEL_LINEUP = get_model_lineup()
for m in MODEL_LINEUP:
    print(f"{m.display_name:35s} {m.kaggle_model_id:30s} {m.role}")


## Step 4 — Model adapter

`tasks.py`'s functions expect any `model` object with a `.generate(prompt) -> response` method where `response.text` is a string — that contract was written generically on purpose. `KaggleModelAdapter` below is where Kaggle's actual model-calling API gets plugged in.

**VERIFY ME:** the body of `KaggleModelAdapter.generate()` is the one piece in this whole repo I can't test from here — Kaggle Benchmarks' exact model invocation call isn't something I can confirm without hands-on access to a live notebook. Wire in the real call once you're inside Kaggle; everything downstream (tasks.py, assertions.py, analysis.py) only needs `.generate(prompt).text` to keep working, so nothing else in the pipeline should need to change.

`MockModel` is a stand-in that needs no Kaggle access at all, so the *rest* of this notebook's wiring can be dry-run and proven correct independently of that one unverified piece.

In [ ]:
import random
from dataclasses import dataclass


@dataclass
class ModelResponse:
    text: str


class KaggleModelAdapter:
    """Thin wrapper around a real Kaggle Benchmarks model.
    VERIFY ME before the real run -- see markdown cell above."""

    def __init__(self, kaggle_model_id: str):
        self.kaggle_model_id = kaggle_model_id
        # TODO: construct whatever Kaggle's SDK actually requires here,
        # e.g. self._client = kbench.load_model(kaggle_model_id)

    def generate(self, prompt: str) -> ModelResponse:
        # TODO: replace with the real call, e.g.
        #   return ModelResponse(text=self._client.generate(prompt))
        raise NotImplementedError(
            "Wire up the real Kaggle model call here before the live run."
        )


class MockModel:
    """Deterministic stand-in so the pipeline can be dry-run and proven
    correct end-to-end without needing Kaggle model access at all."""

    def __init__(self, seed: int = 0):
        self._rng = random.Random(seed)

    def generate(self, prompt: str) -> ModelResponse:
        if "FRAUD or LEGIT" in prompt:
            return ModelResponse(text=self._rng.choice(["FRAUD", "LEGIT", "NOT FRAUD"]))
        # ring-root prompt
        ids_in_prompt = [int(tok) for tok in prompt.split() if tok.isdigit() and len(tok) >= 3]
        if ids_in_prompt and self._rng.random() > 0.3:
            return ModelResponse(text=f"Transaction {self._rng.choice(ids_in_prompt)} looks responsible.")
        return ModelResponse(text="NONE")


# Flip this to False once KaggleModelAdapter is wired up for real.
USE_MOCK = True


## Step 5 — Run the benchmark

Loops over every model x every sample x both tasks, calling straight into the already-tested `tasks.py` functions, and writes `results/raw_responses.csv` in exactly the shape `analysis.py` expects: `model_name, task, seed_txn_id, raw_response`. Only the raw text goes in this file — no parsing happens here, so there is exactly one place (tasks.py / assertions.py) that interprets what a model's answer means.

In [ ]:
import json
import pandas as pd

from src.graph_fraud_benchmark.tasks import classify_transaction, identify_ring_root

with open(DATA_DIR / "serialized_subgraphs.jsonl") as f:
    samples = [json.loads(line) for line in f]

rows = []
for model_spec in MODEL_LINEUP:
    model = MockModel(seed=hash(model_spec.display_name) % 1000) if USE_MOCK \
        else KaggleModelAdapter(model_spec.kaggle_model_id)

    for sample in samples:
        prompt = sample["prompt"]
        seed_id = sample["seed_txn_id"]
        valid_ids = sample["valid_txn_ids"]

        # classify_transaction and identify_ring_root build their own
        # full prompts internally (see tasks.py) -- we pass the
        # serialized subgraph text in, not a finished prompt.
        from src.graph_fraud_benchmark.tasks import build_classification_prompt, build_ring_root_prompt

        classify_resp = model.generate(build_classification_prompt(prompt))
        rows.append({
            "model_name": model_spec.display_name,
            "task": "classify_transaction",
            "seed_txn_id": seed_id,
            "raw_response": classify_resp.text,
        })

        ring_resp = model.generate(build_ring_root_prompt(prompt))
        rows.append({
            "model_name": model_spec.display_name,
            "task": "identify_ring_root",
            "seed_txn_id": seed_id,
            "raw_response": ring_resp.text,
        })

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_DIR / "raw_responses.csv", index=False)
print(f"wrote {len(results_df)} raw responses to {RESULTS_DIR / 'raw_responses.csv'}")
results_df.head()


## Step 6 — Register as an official Kaggle Benchmark

This is the piece that makes the run count as an actual Kaggle Benchmarking Challenge submission rather than just a local script. `tasks.py` already defines `classify_transaction` and `identify_ring_root` with `@kbench.task` (falling back to a no-op decorator when `kbench` isn't installed, which is why Step 5 above works even without it).

**VERIFY ME:** whatever dataset-registration/leaderboard-submission call Kaggle's SDK actually wants (likely something like handing it a DataFrame of samples plus the task functions) isn't something I can confirm without a live notebook. Check Kaggle's current Benchmarks documentation for the exact call and fill it in below — everything it would need (the task functions, the sample prompts, the model lineup) already exists above.

In [ ]:
try:
    import kbench

    # TODO: replace with Kaggle's actual registration call once confirmed, e.g.
    #   kbench.run_benchmark(
    #       tasks=[classify_transaction, identify_ring_root],
    #       samples=samples,
    #       models=[m.kaggle_model_id for m in MODEL_LINEUP],
    #   )
    print("kbench is installed -- wire up the real registration call above.")
except ImportError:
    print("kbench not installed in this environment -- expected when running "
          "outside Kaggle. The dry run above (Step 5) already proved the "
          "pipeline's logic end-to-end independently of this.")


## Step 7 — (Optional) Fill in GNN predictions

`labels.csv`'s `gnn_pred` column is blank by default. `analysis.py` handles that gracefully (it just skips the GNN-agreement columns), so this step is optional -- but filling it in is what unlocks the headline "did the LLM miss what the GNN caught" comparison.

**Known blocker:** the saved test predictions are keyed by `node_index` (0-75), not `TransactionID`, and reconstructing that mapping needs the real `fraud_graph.pkl` -- the copy in the repo zip was a Git LFS pointer stub, not real data. Run `git lfs pull` locally to get the real file, then this cell can be filled in.

In [ ]:
# TODO once fraud_graph.pkl is a real file (not an LFS pointer):
#
# import pickle
# with open(REPO_ROOT / "data/graph/fraud_graph.pkl", "rb") as f:
#     nx_graph = pickle.load(f)
# node_order = list(nx_graph.nodes())  # index i -> node_order[i]
# # then map each node_index in test_predictions.csv back to its
# # "transaction_{TransactionID}" string in node_order, strip the
# # prefix, and fill labels.csv's gnn_pred column by TransactionID.

print("Skipped -- fraud_graph.pkl not yet available (see markdown above).")


## Step 8 — Analyze: build the leaderboard

In [ ]:
from src.graph_fraud_benchmark.analysis import run as analysis_run

analysis_run(
    results_csv=RESULTS_DIR / "raw_responses.csv",
    labels_csv=DATA_DIR / "labels.csv",
    serialized_jsonl=DATA_DIR / "serialized_subgraphs.jsonl",
    raw_transactions_csv=TRANSACTIONS_CSV,
    out_path=RESULTS_DIR / "leaderboard_export.json",
    gnn_threshold=0.1980,  # graph-fraud-ai's own production decision threshold
)


## Step 9 — Next

Leaderboard numbers alone aren't the submission -- the insight is. Take `results/leaderboard_export.json` and write up what it actually shows (especially the `model_caught_what_gnn_missed` / `model_missed_what_gnn_caught` split) in `writeup/devto_submission.md`.